# IMU extraction: validation notebook

Step-by-step check of the extraction pipeline in `src/probing_qc` on one GoPro video.
The production entry point is `scripts/extraction.py`; this notebook shows each stage
and asserts the output is physically sane. See `docs/extraction.md` for background.

Prerequisites: `pip install -e ".[dev]"`, FFmpeg on PATH, and a video in `input/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from probing_qc import extract_sensor, find_gpmf_stream, imu_dataframe, read_gpmf
from probing_qc.gpmf import iter_klv
from probing_qc.telemetry import probe_streams

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VIDEO = ROOT / "input" / "GX010140.MP4"
OUTPUT_CSV = ROOT / "output" / f"{VIDEO.stem}_imu.csv"

assert VIDEO.is_file(), f"Put the video in input/ (or run scripts/download.py): {VIDEO}"
print(f"{VIDEO.name}: {VIDEO.stat().st_size / 1024**3:.2f} GiB")

## 1. Locate the telemetry stream

GoPro stores GPMF telemetry in a data track tagged `gpmd` (handler "GoPro MET").
Its index varies between cameras, so it is looked up instead of hard-coded.

In [ ]:
streams = probe_streams(VIDEO)
display(pd.DataFrame([
    {"index": s["index"], "type": s.get("codec_type"), "tag": s.get("codec_tag_string"),
     "handler": s.get("tags", {}).get("handler_name"), "duration_s": float(s.get("duration", "nan"))}
    for s in streams
]))

gpmf_index = find_gpmf_stream(VIDEO)
video_duration = float(next(s for s in streams if s["codec_type"] == "video")["duration"])
print("GPMF stream index:", gpmf_index, "| video duration:", video_duration, "s")

## 2. Extract the raw GPMF bytes

FFmpeg copies the track to stdout. The bytes are a sequence of `DEVC` containers
(one per ~1 s payload), each holding `STRM` containers per sensor.

In [ ]:
data = read_gpmf(VIDEO, gpmf_index)
print(f"GPMF size: {len(data) / 1024**2:.1f} MiB")

first_devc = next(iter_klv(data))
for entry in first_devc.children():
    if entry.key != "STRM":
        print(entry.key, entry.text() if entry.type == "c" else entry.values().ravel())
        continue
    keys = [c.key for c in entry.children()]
    name = next((c.text() for c in entry.children() if c.key == "STNM"), "")
    print(f"STRM {name!r:40} {keys}")

## 3. Decode accelerometer and gyroscope

Each sensor payload is scaled by its preceding `SCAL`, reordered from the `ORIN`
channel order to camera X/Y/Z, and timestamped from `STMP` (microseconds).

In [ ]:
acc = extract_sensor(data, "ACCL")
gyro = extract_sensor(data, "GYRO")

for s in (acc, gyro):
    t = s.timestamps()
    print(f"{s.key}: {s.num_samples} samples in {len(s.payloads)} payloads, "
          f"ORIN={s.orientation}, units={s.units}, rate={(len(t) - 1) / (t[-1] - t[0]):.2f} Hz")

## 4. Sanity checks

These assertions are the acceptance criteria for an extraction.

In [ ]:
df = imu_dataframe(data)
acc_norm = np.linalg.norm(df[["acc_x", "acc_y", "acc_z"]], axis=1)
duration = df.timestamp_s.iloc[-1] - df.timestamp_s.iloc[0]

checks = {
    "no missing values": not df.isna().any().any(),
    "timestamps strictly increasing": bool((np.diff(df.timestamp_s) > 0).all()),
    "IMU duration matches video (±1 s)": abs(df.timestamp_s.iloc[-1] - video_duration) < 1.0,
    "sample rate 150-250 Hz": 150 < (len(df) - 1) / duration < 250,
    "median |acc| ≈ g (9.81 ± 0.5 m/s²)": abs(np.median(acc_norm) - 9.81) < 0.5,
    "gyro bias < 0.05 rad/s": bool((df[["gyro_x", "gyro_y", "gyro_z"]].mean().abs() < 0.05).all()),
}
display(pd.Series(checks, name="passed"))
assert all(checks.values()), "Sanity check failed"
df.describe().T

## 5. Compare with the script output

Confirms `scripts/extraction.py` wrote the same table (CSV is rounded to 6 decimals).

In [ ]:
if OUTPUT_CSV.is_file():
    saved = pd.read_csv(OUTPUT_CSV)
    assert list(saved.columns) == list(df.columns) and len(saved) == len(df)
    np.testing.assert_allclose(saved.to_numpy(), df.to_numpy(), atol=1e-5)
    print(f"{OUTPUT_CSV.name} matches ({len(saved)} rows)")
else:
    print(f"Run: python scripts/extraction.py {VIDEO.name}")

## 6. Visual check (first 60 s)

In [ ]:
window = df[df.timestamp_s < df.timestamp_s.iloc[0] + 60]
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
window.plot(x="timestamp_s", y=["acc_x", "acc_y", "acc_z"], ax=axes[0], lw=0.6)
window.plot(x="timestamp_s", y=["gyro_x", "gyro_y", "gyro_z"], ax=axes[1], lw=0.6)
axes[0].set_ylabel("m/s²")
axes[1].set_ylabel("rad/s")
axes[1].set_xlabel("time since recording start (s)")
plt.tight_layout()